# Module 00, Notebook 2: First Causal Analysis with CausalPy

## Learning Objectives
By completing this notebook, you will:
1. Load and explore a real policy intervention dataset
2. Run a complete Interrupted Time Series analysis with CausalPy
3. Visualize the counterfactual trajectory and causal impact
4. Interpret the posterior distribution of the treatment effect
5. Articulate what the ITS model assumes and what it estimates

## Prerequisites
- Completed Notebook 01 (environment setup working)
- Read Guides 1–3 (causal thinking, potential outcomes, DAGs)

## Estimated Time: 15 minutes

---

## 1. The Analysis Question

We analyze the effect of a workplace wellness program on employee productivity, measured by weekly output units.

**Dataset:** Monthly productivity data for a manufacturing division. A structured wellness intervention (ergonomic improvements, stress reduction workshops, flexible hours policy) was implemented in month 25.

**Causal question:** Did the wellness program change the trajectory of employee productivity?

**ITS design:** We use the pre-intervention trend as the counterfactual. We estimate both:
- The **level change** at month 25 (immediate effect)
- The **slope change** after month 25 (sustained or growing effect)

This dataset is realistic — it mirrors patterns seen in workplace intervention studies, with a genuine pre-trend, noise, and a plausible causal effect.

In [ ]:
# Standard imports for causal analysis
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import causalpy as cp
import arviz as az

np.random.seed(42)
plt.style.use("seaborn-v0_8-whitegrid")
%matplotlib inline

print("Imports successful.")

## 2. Load and Explore the Dataset

The dataset contains 48 months of productivity data for a manufacturing division. Month 25 is when the wellness program launched.

In [ ]:
# Generate the workplace wellness dataset
# This mimics real workplace intervention data with:
# - A realistic pre-trend (slight positive drift as workers gain experience)
# - Seasonal patterns (dip in summer months, peak in Q4)
# - Realistic noise levels
# - A genuine treatment effect: +8 units/month level change, +0.3 units/month slope increase

np.random.seed(2024)
n_months = 48
intervention_month = 24  # 0-indexed, so month 25 in human terms

months = np.arange(n_months)
# Calendar months for seasonal pattern
calendar_month = months % 12

# Seasonal effect: slightly lower in months 6-8 (summer), higher in months 10-11 (Q4)
seasonal_effect = (
    -3.0 * np.sin(2 * np.pi * calendar_month / 12)
    - 2.0 * np.cos(2 * np.pi * calendar_month / 12)
)

# Treatment indicators
treated = (months >= intervention_month).astype(float)
months_since_intervention = np.maximum(months - intervention_month, 0).astype(float)

# True data-generating process
# Baseline: 100 units/month starting productivity
# Pre-trend: +0.8 units/month (experience accumulation)
# Level change: +8 units immediately after intervention
# Slope change: +0.3 additional units/month growth after intervention
true_level_change = 8.0
true_slope_change = 0.3

baseline = 100.0
pre_trend = 0.8
noise_std = 4.5

productivity = (
    baseline
    + pre_trend * months
    + true_level_change * treated
    + true_slope_change * months_since_intervention * treated
    + seasonal_effect
    + np.random.normal(0, noise_std, n_months)
)

# Build the DataFrame
df = pd.DataFrame({
    "month": months,
    "productivity": productivity,
    "treated": treated,
    "months_since_intervention": months_since_intervention,
    "calendar_month": calendar_month,
})

# Add a human-readable date column (starting 2022-01)
df["date"] = pd.date_range(start="2022-01", periods=n_months, freq="MS")

print("Dataset created.")
print(f"Shape: {df.shape}")
print(f"Pre-intervention months: {(df['treated'] == 0).sum()}")
print(f"Post-intervention months: {(df['treated'] == 1).sum()}")
print(f"Intervention date: {df.loc[df['treated'] == 1, 'date'].min().strftime('%Y-%m')}")
print(f"\nProductivity statistics:")
print(df[['productivity', 'treated']].groupby('treated').describe()['productivity'].round(2))

In [ ]:
# Visualize the raw data before fitting any model
# Always look at your data first — models should explain what you see

fig, ax = plt.subplots(figsize=(12, 5))

# Pre-intervention data
pre_mask = df["treated"] == 0
post_mask = df["treated"] == 1

ax.plot(
    df.loc[pre_mask, "date"],
    df.loc[pre_mask, "productivity"],
    "o-",
    color="#2c3e50",
    label="Pre-intervention",
    linewidth=1.5,
    markersize=4,
)
ax.plot(
    df.loc[post_mask, "date"],
    df.loc[post_mask, "productivity"],
    "s-",
    color="#27ae60",
    label="Post-intervention",
    linewidth=1.5,
    markersize=4,
)

# Mark intervention date
intervention_date = df.loc[intervention_month, "date"]
ax.axvline(
    intervention_date,
    color="#e74c3c",
    linestyle="--",
    linewidth=2,
    label=f"Wellness program launch ({intervention_date.strftime('%Y-%m')})",
)

ax.set_xlabel("Date", fontsize=12)
ax.set_ylabel("Monthly Productivity (units)", fontsize=12)
ax.set_title("Employee Productivity Before and After Wellness Program", fontsize=14)
ax.legend(fontsize=11)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("\nWhat do you notice in this plot?")
print("  - Is there a visible level change at the intervention?")
print("  - Does the slope appear to change after the intervention?")
print("  - What seasonal patterns do you see?")

## 3. Specify the ITS Model

The standard segmented regression model for ITS includes:
- `month`: the secular time trend (pre-existing trajectory)
- `treated`: the **level change** at the intervention point (immediate effect)
- `months_since_intervention`: the **slope change** after the intervention (sustained effect)

Formally:
$$Y_t = \alpha + \beta_1 t + \beta_2 \cdot \text{treated}_t + \beta_3 \cdot (t - t^*)^+ + \varepsilon_t$$

Where $(t - t^*)^+ = \max(t - t^*, 0)$ is the months since the intervention (zero in the pre-period).

In [ ]:
# Fit the ITS model using CausalPy
# The formula directly maps to the segmented regression equation above

its_result = cp.InterruptedTimeSeries(
    data=df,
    treatment_time=intervention_month,
    formula="productivity ~ 1 + month + treated + months_since_intervention",
    model=cp.pymc_models.LinearRegression(
        sample_kwargs={
            "draws": 1000,
            "tune": 1000,
            "chains": 4,
            "progressbar": True,
            "random_seed": 42,
        }
    ),
)

print("\nITS model fitted successfully.")

## 4. Visualize the Results

CausalPy provides a built-in plot that shows:
- Observed data points
- Fitted model (with uncertainty bands)
- Counterfactual trajectory (what would have happened without the intervention)
- Causal impact (the difference between observed and counterfactual)

In [ ]:
# CausalPy's built-in visualization
# Top panel: observed data + fitted model + counterfactual
# Bottom panel: estimated causal impact at each post-intervention time point

fig, axes = its_result.plot()
axes[0].set_title("ITS: Wellness Program Impact on Productivity", fontsize=13)
axes[0].set_xlabel("Month index")
axes[0].set_ylabel("Monthly Productivity (units)")
axes[1].set_ylabel("Causal Impact (units/month)")
axes[1].set_xlabel("Month index")
plt.tight_layout()
plt.show()

## 5. Interpret the Posterior Coefficients

The Bayesian model gives us posterior distributions for each coefficient:
- `Intercept (α)`: baseline productivity at t=0
- `month (β₁)`: monthly growth rate in pre-intervention period
- `treated (β₂)`: **immediate level change** at the intervention
- `months_since_intervention (β₃)`: additional monthly growth after intervention

In [ ]:
# Plot posterior distributions of all model coefficients
# The 94% HDI (Highest Density Interval) is the Bayesian analog of a confidence interval

az.plot_posterior(
    its_result.idata,
    var_names=["Intercept", "month", "treated", "months_since_intervention"],
    ref_val=0,  # Show reference line at zero
    figsize=(14, 4),
)
plt.suptitle("Posterior Distributions of ITS Model Coefficients", y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Extract and print the key causal estimates
# For policy communication, we report: posterior mean, 94% HDI, P(effect > 0)

posterior = its_result.idata.posterior

def summarize_posterior(samples, name, true_value=None):
    """Print a summary of a posterior distribution."""
    flat = samples.values.flatten()
    mean = flat.mean()
    hdi = az.hdi(flat, hdi_prob=0.94)
    prob_positive = (flat > 0).mean()
    
    print(f"  {name}:")
    print(f"    Posterior mean:    {mean:.2f}")
    print(f"    94% HDI:           [{hdi[0]:.2f}, {hdi[1]:.2f}]")
    print(f"    P(effect > 0):     {prob_positive:.1%}")
    if true_value is not None:
        print(f"    True value:        {true_value:.2f}")
    print()

print("Causal Effect Estimates")
print("=" * 50)
summarize_posterior(posterior["treated"], "Immediate level change (β₂)", true_level_change)
summarize_posterior(posterior["months_since_intervention"], "Additional monthly growth (β₃)", true_slope_change)

print("Interpretation:")
print("  β₂ = the immediate jump in productivity when the program launched")
print("  β₃ = how much faster productivity grew each month after the program")

In [ ]:
# Compute cumulative causal impact over the post-intervention period
# This is the total gain attributable to the wellness program

# Extract posterior samples of the causal impact at each post-intervention time
# The causal impact at time t is: Y(1) - Y(0) = observed - counterfactual

beta_level = posterior["treated"].values.flatten()
beta_slope = posterior["months_since_intervention"].values.flatten()

# Post-intervention months (0 to 23)
t_post_values = np.arange(0, n_months - intervention_month)

# Causal impact at each post-intervention time point (posterior distribution)
# Impact at month k: beta_level + beta_slope * k
n_samples = len(beta_level)
impact_matrix = beta_level[:, np.newaxis] + beta_slope[:, np.newaxis] * t_post_values[np.newaxis, :]

# Cumulative impact over all post-intervention months
cumulative_impact = impact_matrix.sum(axis=1)

# Summarize
print("Cumulative Causal Impact (total over post-intervention period):")
print(f"  Posterior mean: {cumulative_impact.mean():.1f} units")
hdi_cum = az.hdi(cumulative_impact, hdi_prob=0.94)
print(f"  94% HDI: [{hdi_cum[0]:.1f}, {hdi_cum[1]:.1f}] units")
print(f"  P(positive impact): {(cumulative_impact > 0).mean():.1%}")

# Plot the posterior of cumulative impact
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(cumulative_impact, bins=60, edgecolor="white", color="#3498db", alpha=0.8)
ax.axvline(0, color="red", linestyle="--", linewidth=2, label="Zero effect")
ax.axvline(cumulative_impact.mean(), color="#2c3e50", linewidth=2, label=f"Posterior mean: {cumulative_impact.mean():.0f}")
ax.set_xlabel("Cumulative Productivity Gain (units)", fontsize=12)
ax.set_ylabel("Posterior Density", fontsize=12)
ax.set_title("Posterior Distribution: Total Productivity Gain from Wellness Program", fontsize=13)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

## 6. Check Convergence

Before trusting the results, verify that the MCMC sampler converged. R-hat values should be below 1.01 and effective sample sizes should be large.

In [ ]:
# Check MCMC convergence diagnostics
# R-hat: measures chain mixing. Values < 1.01 indicate good convergence.
# ESS: effective sample size. Should be > 400 for reliable estimates.

print("MCMC Convergence Diagnostics")
print("=" * 50)

summary = az.summary(
    its_result.idata,
    var_names=["Intercept", "month", "treated", "months_since_intervention", "sigma"],
)
print(summary[["mean", "sd", "hdi_3%", "hdi_97%", "r_hat", "ess_bulk"]].to_string())

print("\nDiagnostic interpretation:")
print("  r_hat < 1.01: Good convergence")
print("  ess_bulk > 400: Sufficient effective samples")

# Plot trace plots for visual convergence check
az.plot_trace(
    its_result.idata,
    var_names=["treated", "months_since_intervention"],
    figsize=(12, 4),
)
plt.suptitle("Trace Plots: MCMC Chain Mixing", y=1.02)
plt.tight_layout()
plt.show()

## 7. Communicating the Results

A complete ITS analysis report includes three components:
1. **What the model estimates** (effect sizes with uncertainty)
2. **Why we believe the estimate is causal** (assumption justification)
3. **What alternative explanations remain** (threats to validity)

Here is the structure for communicating these results:

In [ ]:
# Generate a structured results summary for communication

beta_level_mean = posterior["treated"].values.flatten().mean()
beta_level_hdi = az.hdi(posterior["treated"].values.flatten(), hdi_prob=0.94)
beta_slope_mean = posterior["months_since_intervention"].values.flatten().mean()
beta_slope_hdi = az.hdi(posterior["months_since_intervention"].values.flatten(), hdi_prob=0.94)
prob_positive_level = (posterior["treated"].values.flatten() > 0).mean()

print("\n" + "=" * 60)
print("ANALYSIS RESULTS SUMMARY")
print("Wellness Program Impact on Employee Productivity")
print("=" * 60)
print()
print("DESIGN")
print(f"  Method: Interrupted Time Series (ITS)")
print(f"  Pre-period: {intervention_month} months")
print(f"  Post-period: {n_months - intervention_month} months")
print(f"  Estimation: Bayesian linear regression (PyMC/CausalPy)")
print()
print("MAIN FINDINGS")
print(f"  Immediate productivity gain: {beta_level_mean:.1f} units/month")
print(f"  94% Credible Interval: [{beta_level_hdi[0]:.1f}, {beta_level_hdi[1]:.1f}]")
print(f"  Probability of positive effect: {prob_positive_level:.1%}")
print()
print(f"  Additional monthly growth rate: {beta_slope_mean:.2f} units/month")
print(f"  94% Credible Interval: [{beta_slope_hdi[0]:.2f}, {beta_slope_hdi[1]:.2f}]")
print()
print(f"  Cumulative gain (24 months): ~{cumulative_impact.mean():.0f} units")
print()
print("ASSUMPTION JUSTIFICATION")
print("  (1) Pre-trend extrapolation: Assumes productivity would have")
print("      continued its pre-program linear growth trajectory.")
print("  (2) No concurrent events: No other major workplace changes")
print("      occurred at month 25.")
print("  (3) No anticipation effects: Program announced and launched")
print("      simultaneously.")
print()
print("THREATS TO VALIDITY")
print("  - Unobserved concurrent changes (economic conditions, workforce changes)")
print("  - Possible regression to the mean if intervention triggered by dip")
print("  - Non-linearity in pre-trend not captured by linear model")
print("=" * 60)

## Summary

### What You Learned
1. **ITS workflow:** Data exploration → Model specification → Sampling → Interpretation
2. **CausalPy API:** `InterruptedTimeSeries` handles the full Bayesian ITS pipeline
3. **Causal interpretation:** The `treated` coefficient is the level change; `months_since_intervention` is the slope change
4. **Bayesian advantage:** We get full posterior distributions over the counterfactual and causal impact, not just point estimates
5. **Convergence checking:** R-hat and ESS diagnostics verify the sampler worked correctly

### The ITS Estimand
We estimated the **ATT (Average Treatment Effect on the Treated)** — the effect of the wellness program on the division that actually received it, compared to what would have happened had the program not been implemented.

### Key Assumption
The pre-intervention trend would have continued absent the program. The credibility of this assumption depends on:
- No concurrent workplace changes at month 25
- No anticipation effects before month 25
- The linear model capturing the true pre-trend shape

### What's Next
In **Module 01**, we formalize the ITS methodology:
- When ITS is (and is not) appropriate
- How to handle seasonality, autocorrelation, and non-linear trends
- CausalPy's `InterruptedTimeSeries` API in depth